# Metrics — JSD, HD and RMAD

Input: `Appendix_E_Elicitation_Data.xlsx`.  
Output: `metrics.csv` (Tables 2–7).

In [ ]:
import numpy as np
import pandas as pd

FILE = "Appendix_E_Elicitation_Data.xlsx"
MODELS = ["ChatGPT", "Gemini", "Claude", "LLaMA", "DeepSeek"]

## 1. Load data

In [ ]:
ess = pd.read_excel(FILE, sheet_name="E_ESS_observed")
llm = pd.read_excel(FILE, sheet_name="E_LLM_outputs")
ref = pd.read_excel(FILE, sheet_name="E_metrics")
ess.head()

## 2. Metrics

In [ ]:
def jsd(p, q):
    m = (p + q) / 2
    kl = lambda a, b: np.sum(a[a > 0] * np.log(a[a > 0] / b[a > 0]))  # natural log
    return np.sqrt(0.5 * kl(p, m) + 0.5 * kl(q, m))

def hd(p, q):
    return np.sqrt(np.sum((np.sqrt(p) - np.sqrt(q)) ** 2)) / np.sqrt(2)

def rmad(p, q):
    return np.mean(np.abs(p - q))

In [ ]:
rows = []
for (q, country), g in ess.groupby(["question", "country"], sort=False):
    Q = g.sort_values("response_code")["proportion"].to_numpy()
    for m in MODELS:
        P = llm[(llm.question == q) & (llm.country == country) & (llm.model == m)] \
            .sort_values("response_code")["proportion"].to_numpy()
        rows.append({"question": q, "country": country, "model": m,
                     "JSD": round(jsd(P, Q), 4), "HD": round(hd(P, Q), 4),
                     "RMAD_pct": round(rmad(P, Q) * 100, 2)})
metrics = pd.DataFrame(rows)
metrics.to_csv("metrics.csv", index=False)
metrics

## 3. Comparison with sheet E_metrics

In [ ]:
chk = metrics.merge(ref, on=["question", "country", "model"], suffixes=("", "_ref"))
print("Comparisons:", len(chk))
print("Max difference:", max((chk[k] - chk[k + "_ref"]).abs().max() for k in ["JSD", "HD", "RMAD_pct"]))